# Neural Networks from Zero to World Cup Predictions
## A Beginner's Step-by-Step Tutorial

---

Welcome! This notebook will take you from **zero knowledge** about neural networks all the way to building a model that predicts **FIFA World Cup match outcomes**.

No machine learning background is needed — only basic Python knowledge.

### What you will learn:
1. What a neural network is (in plain English)
2. How neural networks learn from data
3. What training data is and how to prepare it
4. How to make predictions
5. How to measure and improve accuracy

### What you will build:
A complete World Cup match predictor that takes two teams and estimates who wins.

---

## How to Use This Notebook
- Read every explanation **before** running the code
- Run each cell in order (top to bottom) by pressing **Shift + Enter**
- Don't skip cells — later code depends on earlier code
- Every code block has comments explaining what each line does

---
# PART 1: What Is a Neural Network?

## The Brain Analogy

Your brain has about **86 billion neurons** (nerve cells). Each neuron receives signals from other neurons, processes them, and either fires a signal forward or stays quiet. This is how you recognize a face, learn to ride a bike, or know that Brazil vs Germany is a big match.

A **neural network** in a computer works the same way — but much simpler:

```
Input Layer        Hidden Layer       Output Layer
  (data in)       (processing)        (answer out)

 [Team A rank] →  [Neuron 1] \              
                  [Neuron 2] → [Neuron A] → [Win/Lose/Draw]
 [Team B rank] →  [Neuron 3] /              
```

**Three kinds of layers:**
- **Input layer**: Where your data enters. Like eyes or ears for the brain.
- **Hidden layers**: Where the real work happens. Like the "thinking" part.
- **Output layer**: Where the answer comes out. Like your brain deciding yes or no.

## Neurons and Weights

Each connection between neurons has a **weight** — a number that says how important that connection is.

Think of it like a recipe:
- `FIFA ranking` × 0.8  (very important)
- `Goals scored last year` × 0.3  (somewhat important)
- `Home advantage` × 0.1  (a little important)

The neural network **learns the right weights** by looking at past match results.

## Activation Functions — The On/Off Switch

After adding up all the weighted inputs, a neuron applies an **activation function** to decide:
> "Should I be excited (pass a strong signal) or calm (pass a weak one)?"

The most common one is **ReLU**: if the number is negative, output 0. If positive, pass it through.

```
ReLU(-3) = 0
ReLU(5)  = 5
```

---
# STEP 1: Setting Up Your Environment

Before we write any neural network code, we need to import the tools (libraries) that do the heavy lifting.

### Why each library?
| Library | What it does |
|---------|-------------|
| `numpy` | Fast math on arrays of numbers |
| `pandas` | Work with tables of data (like Excel) |
| `matplotlib` | Draw charts and graphs |
| `seaborn` | Beautiful statistical charts |
| `sklearn` | Pre-built machine learning tools |

Run the cell below to import everything:

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings                      # Suppress noisy warnings
warnings.filterwarnings('ignore')    # Keep the output clean

# ── Numerical computing ────────────────────────────────────────────────────────
import numpy as np                   # NumPy: math on arrays

# ── Data manipulation ──────────────────────────────────────────────────────────
import pandas as pd                  # Pandas: spreadsheet-style tables

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt      # Matplotlib: basic plotting
import matplotlib.patches as mpatches
import seaborn as sns                # Seaborn: prettier statistical plots

# ── Machine learning ───────────────────────────────────────────────────────────
from sklearn.neural_network import MLPClassifier   # Our neural network
from sklearn.preprocessing import StandardScaler  # Normalise numbers
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Reproducibility ────────────────────────────────────────────────────────────
np.random.seed(42)                   # Fix randomness so results are consistent

# ── Styling ────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

print("All libraries imported successfully!")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")
print("Ready to build neural networks!")

---
# PART 2: How Neural Networks Learn

## The Learning Loop (in plain English)

Imagine you're learning to throw darts. Here's what happens:

1. **Throw** — You make a guess (forward pass)
2. **Miss** — You see how far off you were (calculate loss/error)
3. **Adjust** — You change your throw slightly to do better (backpropagation)
4. **Repeat** — You keep practising until you hit the bullseye consistently (training epochs)

Neural networks do exactly this, thousands of times, with numbers.

## Key Terms

| Term | Plain English |
|------|---------------|
| **Epoch** | One complete pass through all training data |
| **Loss** | A number measuring how wrong the network is (lower = better) |
| **Backpropagation** | Algorithm that figures out how to adjust each weight to reduce loss |
| **Learning rate** | How big each adjustment step is (too big = overshoot, too small = slow) |
| **Gradient descent** | The strategy of always adjusting weights toward lower loss |

## Visualising Loss Over Time

Below we simulate what a training loss curve looks like. As the network trains, the loss drops:

In [ ]:
# Simulate a training loss curve to understand what 'learning' looks like

epochs = np.arange(1, 101)           # 100 training rounds

# Simulate loss: starts high, drops fast, then levels off (with noise)
loss = 1.0 / (0.1 * epochs) + np.random.normal(0, 0.02, 100)
loss = np.clip(loss, 0.05, None)     # Loss can't go below 0.05 in this demo

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, loss, color='steelblue', linewidth=2)
ax.fill_between(epochs, loss, alpha=0.15, color='steelblue')
ax.set_xlabel('Epoch (training round)', fontsize=12)
ax.set_ylabel('Loss (how wrong the model is)', fontsize=12)
ax.set_title('How a Neural Network Learns: Loss Decreasing Over Time', fontsize=14)

# Annotate key phases
ax.annotate('Early training:\nhigh error', xy=(5, loss[4]),
            xytext=(20, 0.6), fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))
ax.annotate('Late training:\nlow error', xy=(90, loss[89]),
            xytext=(65, 0.3), fontsize=10, color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

plt.tight_layout()
plt.show()

print("Notice how the loss drops quickly at first, then gradually flattens out.")
print("That flattening means the network has learned as much as it can from the data.")

---
# PART 3: Build a Single Neuron from Scratch

Before using a library, let's build the simplest possible neuron by hand so you truly understand what's happening inside.

## What a single neuron computes:

```
output = activation( weight1*input1 + weight2*input2 + bias )
```

- **Weights** — how much each input matters
- **Bias** — a constant that shifts the result (like an intercept in y = mx + b)
- **Activation** — squashes the result into a useful range

### Example: Will Brazil win based on two numbers?
- Input 1: FIFA ranking difference (positive = Brazil is ranked higher)
- Input 2: Goals per game difference

In [ ]:
# ── BUILD A SINGLE NEURON FROM SCRATCH ────────────────────────────────────────

def sigmoid(x):
    """Squash any number into the range [0, 1].
    Output close to 1 means 'yes/win', close to 0 means 'no/lose'."""
    return 1 / (1 + np.exp(-x))

def single_neuron(input1, input2, weight1, weight2, bias):
    """One neuron: multiply inputs by weights, add bias, apply activation."""
    # Step 1: Weighted sum of all inputs
    weighted_sum = (weight1 * input1) + (weight2 * input2) + bias

    # Step 2: Apply activation function (sigmoid gives probability)
    output = sigmoid(weighted_sum)
    return weighted_sum, output

# ── Manually set weights (a real network learns these automatically) ────────────
w1 = 0.5    # Weight for ranking difference
w2 = 0.8    # Weight for goals-per-game difference
b  = 0.1    # Bias

# ── Test cases ─────────────────────────────────────────────────────────────────
test_cases = [
    ("Brazil vs Germany",    2.0,  0.3),   # Brazil ranked much higher, slightly better attack
    ("Germany vs Brazil",   -2.0, -0.3),   # Germany ranked lower against Brazil
    ("Argentina vs France",  0.5,  0.1),   # Very close match
    ("San Marino vs Brazil",-5.0, -2.0),   # Huge gap
]

print(f"{'Match':<30} {'Rank Diff':>10} {'Goal Diff':>10} {'Raw Sum':>10} {'Win Prob':>10} {'Prediction':>12}")
print("-" * 86)

for match_name, rank_diff, goal_diff in test_cases:
    raw, prob = single_neuron(rank_diff, goal_diff, w1, w2, b)
    prediction = "WIN" if prob > 0.5 else "LOSE/DRAW"
    print(f"{match_name:<30} {rank_diff:>10.1f} {goal_diff:>10.1f} {raw:>10.3f} {prob:>10.3f} {prediction:>12}")

print("\nKey insight: sigmoid(x) always outputs between 0 and 1, perfect for probabilities!")

---
# PART 4: What Is Training Data?

## The Concept

**Training data** is a collection of past examples with known answers.

For a football predictor:
- **Input (features)**: Team rankings, goals scored, head-to-head record, etc.
- **Output (label)**: Who actually won (1 = Team A won, 0 = Team B won, 0.5 = draw)

Think of it like a textbook with worked examples:
> "When X happened, the result was Y."

The network studies thousands of these examples and learns the patterns.

## Training vs Test Data

We always split data into two parts:
- **Training set (80%)**: What the network learns from. Like your study notes.
- **Test set (20%)**: What we use to check how well it learned. Like the final exam.

**Why not use the same data for both?** Because a student who memorises the exact questions will ace the test but fail at new questions. We want the network to *generalise*, not *memorise*.

This problem of memorising instead of learning is called **overfitting**.

## Feature Engineering

**Features** are the input numbers we give the neural network. The better your features, the better your predictions.

For football, useful features include:
- FIFA ranking of each team
- Average goals scored/conceded
- Wins/losses in recent matches
- Head-to-head history
- Tournament stage (group stage vs final)
- Whether a team is playing on home continent

---
# PART 5: Warm-Up Example — Predicting Pass/Fail

Before tackling football, let's train a tiny neural network on a dead-simple problem: predicting whether a student **passes** or **fails** based on hours studied and sleep.

This lets you understand the full training loop without any football complexity.

In [ ]:
# ── CREATE SIMPLE TRAINING DATA ────────────────────────────────────────────────
# Each row: [hours_studied, hours_slept], label: 1=pass, 0=fail

np.random.seed(42)

n_students = 200

# Generate study hours (1-10) and sleep hours (3-10)
hours_studied = np.random.uniform(1, 10, n_students)
hours_slept   = np.random.uniform(3, 10, n_students)

# A student passes if: study_score + sleep_score is high enough (with some noise)
score = 0.6 * hours_studied + 0.4 * hours_slept + np.random.normal(0, 0.5, n_students)
labels = (score > 6.5).astype(int)   # 1 = pass, 0 = fail

# Put into a DataFrame (like a spreadsheet)
df_students = pd.DataFrame({
    'hours_studied': hours_studied,
    'hours_slept':   hours_slept,
    'passed':        labels
})

print("First 8 rows of training data:")
print(df_students.head(8).to_string(index=False))
print(f"\nTotal students: {n_students}")
print(f"Passed: {labels.sum()} | Failed: {(labels==0).sum()}")

In [ ]:
# Visualise the data before training

fig, ax = plt.subplots(figsize=(8, 6))

# Scatter plot: colour = pass/fail
colors = ['#e74c3c' if l == 0 else '#2ecc71' for l in labels]
ax.scatter(hours_studied, hours_slept, c=colors, alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

ax.set_xlabel('Hours Studied', fontsize=12)
ax.set_ylabel('Hours Slept', fontsize=12)
ax.set_title('Student Data: Will They Pass?', fontsize=14)

pass_patch = mpatches.Patch(color='#2ecc71', label='Pass')
fail_patch = mpatches.Patch(color='#e74c3c', label='Fail')
ax.legend(handles=[pass_patch, fail_patch], fontsize=11)

plt.tight_layout()
plt.show()

print("Notice the pattern: students who study more AND sleep more tend to pass.")
print("The neural network will discover this pattern automatically.")

In [ ]:
# ── STEP-BY-STEP: Train a Neural Network ──────────────────────────────────────

# STEP A: Separate features (inputs) from labels (correct answers)
X = df_students[['hours_studied', 'hours_slept']].values   # 2D input matrix
y = df_students['passed'].values                           # 1D label array

print("Feature matrix shape:", X.shape, "  (200 students, 2 features each)")
print("Label array shape   :", y.shape, "  (200 pass/fail answers)")
print()

# STEP B: Split into training set and test set
# test_size=0.2 means 20% (40 students) held back for testing
# random_state=42 makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training examples : {len(X_train)}  (network will learn from these)")
print(f"Test examples     : {len(X_test)}   (we'll use these to check accuracy)")
print()

# STEP C: Normalise (scale) the features
# Neural networks learn MUCH faster when all input numbers are in a similar range.
# StandardScaler makes each feature have mean=0 and std=1.
# Rule: ALWAYS fit the scaler on training data only, then transform both sets.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Learn mean/std from training
X_test_scaled  = scaler.transform(X_test)        # Apply SAME scaling to test

print("Before scaling - first training example:", X_train[0])
print("After  scaling - first training example:", X_train_scaled[0].round(3))
print("(Numbers now centred around 0 — easier for the network to learn)")

In [ ]:
# ── STEP D: Define and Train the Neural Network ────────────────────────────────

# MLPClassifier = Multi-Layer Perceptron Classifier
# This IS the neural network.
simple_nn = MLPClassifier(
    hidden_layer_sizes=(8, 4),   # 2 hidden layers: first has 8 neurons, second has 4
    activation='relu',           # ReLU activation function
    max_iter=500,                # Maximum training rounds (epochs)
    learning_rate_init=0.01,     # How big each adjustment step is
    random_state=42,             # For reproducibility
    verbose=False                # Don't print every epoch
)

# .fit() is where the learning happens!
# The network repeatedly sees the training data and adjusts its weights.
simple_nn.fit(X_train_scaled, y_train)

print("Training complete!")
print(f"Network architecture: Input(2) -> Hidden(8) -> Hidden(4) -> Output(1)")
print(f"Total training iterations: {simple_nn.n_iter_}")

In [ ]:
# ── STEP E: Evaluate the Trained Network ──────────────────────────────────────

# Make predictions on the TEST set (data the network has never seen)
y_pred = simple_nn.predict(X_test_scaled)

# Accuracy = fraction of correct predictions
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.1%}")
print()

# Detailed breakdown
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

# The report shows:
# precision  = of all predicted 'pass', how many actually passed
# recall     = of all actual passes, how many did we correctly find
# f1-score   = balanced average of precision and recall

In [ ]:
# ── Confusion Matrix: The 2×2 Report Card ─────────────────────────────────────
#
# A confusion matrix shows exactly where the model is right and wrong:
#
#                   Predicted Fail  |  Predicted Pass
#  Actual Fail:   True Negative (TN)|  False Positive (FP)  <- said pass but failed
#  Actual Pass:   False Negative(FN)|  True Positive  (TP)  <- correctly identified pass

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Student Pass/Fail Predictor', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (correctly predicted PASS): {tp}")
print(f"True Negatives  (correctly predicted FAIL): {tn}")
print(f"False Positives (predicted PASS, was FAIL): {fp}  <- Type I error")
print(f"False Negatives (predicted FAIL, was PASS): {fn}  <- Type II error")

In [ ]:
# ── Visualise the Decision Boundary ───────────────────────────────────────────
# This shows WHERE the neural network draws the line between Pass and Fail.

# Create a fine grid covering the input space
h = 0.05
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict for every point on the grid
grid_points = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
Z = simple_nn.predict(grid_points).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 7))
ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
ax.contour(xx, yy, Z, colors='black', linewidths=1, alpha=0.4)

# Plot actual data points on top
colors = ['#e74c3c' if l == 0 else '#27ae60' for l in y]
ax.scatter(X[:, 0], X[:, 1], c=colors, alpha=0.8, s=50, edgecolors='white', linewidth=0.5)

ax.set_xlabel('Hours Studied', fontsize=12)
ax.set_ylabel('Hours Slept',   fontsize=12)
ax.set_title('Neural Network Decision Boundary\n(Green = Predict Pass, Red = Predict Fail)', fontsize=13)

pass_patch = mpatches.Patch(color='#27ae60', label='Pass')
fail_patch = mpatches.Patch(color='#e74c3c', label='Fail')
ax.legend(handles=[pass_patch, fail_patch], fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

print("The network has learned a curved boundary — something a simple straight-line")
print("rule couldn't do. This is the power of neural networks!")

---
# PART 6: The World Cup Prediction Project

Now that you understand the basics, let's build the real project: **predicting FIFA World Cup match outcomes**.

## Our Prediction Goal

Given two teams playing each other, predict one of three outcomes:
- **Team 1 Win** (label = 0)
- **Draw** (label = 1)  
- **Team 2 Win** (label = 2)

## Features We Will Use

| Feature | What it measures |
|---------|------------------|
| `ranking_diff` | Difference in FIFA ranking (positive = team 1 is higher ranked) |
| `goals_scored_avg_diff` | Difference in average goals scored per game |
| `goals_conceded_avg_diff` | Difference in average goals conceded per game |
| `win_rate_diff` | Difference in recent win rates |
| `head_to_head_wins` | How many times team 1 beat team 2 historically |
| `tournament_stage` | Group stage (0), Round of 16 (1), QF (2), SF (3), Final (4) |
| `confederation_advantage` | 1 if team 1 plays on home continent, -1 if team 2 does, 0 if neither |
| `avg_player_age_diff` | Difference in squad average age |

## Data Strategy

We will create a realistic synthetic dataset based on known patterns from World Cup history. This ensures we can teach the full pipeline without requiring you to download external files. The data is structured to match real-world distributions.

In [ ]:
# ── GENERATE REALISTIC WORLD CUP TRAINING DATA ────────────────────────────────
# We build ~1500 matches that mirror real World Cup statistics.

np.random.seed(42)
n_matches = 1500

def generate_world_cup_data(n):
    """Generate synthetic but realistic World Cup match data."""

    # Feature 1: FIFA ranking difference (Team1_rank - Team2_rank)
    # Positive means Team 1 is ranked higher (better team)
    ranking_diff = np.random.normal(0, 30, n)           # Mean 0, spread of 30 places

    # Feature 2: Average goals scored per game difference
    goals_scored_diff = np.random.normal(0, 0.8, n)

    # Feature 3: Average goals conceded per game difference (negative = team 1 better defence)
    goals_conceded_diff = np.random.normal(0, 0.6, n)

    # Feature 4: Recent win rate difference (-1 to +1)
    win_rate_diff = np.random.uniform(-0.6, 0.6, n)

    # Feature 5: Head-to-head wins for Team 1 out of last 5 meetings (0-5)
    h2h_wins = np.random.randint(0, 6, n)

    # Feature 6: Tournament stage (0=group, 1=R16, 2=QF, 3=SF, 4=Final)
    stage = np.random.choice([0, 1, 2, 3, 4], n, p=[0.48, 0.24, 0.12, 0.10, 0.06])

    # Feature 7: Confederation advantage (-1, 0, +1)
    conf_adv = np.random.choice([-1, 0, 1], n, p=[0.15, 0.70, 0.15])

    # Feature 8: Average player age difference
    age_diff = np.random.normal(0, 1.5, n)

    # ── Generate realistic outcomes ────────────────────────────────────────────
    # A higher-ranked team with better recent form wins more often.
    # We model the probability of each outcome based on the features.

    outcomes = []
    for i in range(n):
        # Compute a 'strength score' for Team 1
        strength = (
            0.030 * ranking_diff[i]
            + 0.400 * goals_scored_diff[i]
            - 0.300 * goals_conceded_diff[i]
            + 0.500 * win_rate_diff[i]
            + 0.100 * (h2h_wins[i] - 2.5)      # Centred around 2.5
            + 0.150 * conf_adv[i]
            + 0.020 * age_diff[i]
            + np.random.normal(0, 0.5)          # Random noise (football is unpredictable!)
        )

        # Convert strength score to outcome label
        if strength > 0.7:    outcomes.append(0)   # Team 1 win
        elif strength > -0.7: outcomes.append(1)   # Draw
        else:                 outcomes.append(2)   # Team 2 win

    return pd.DataFrame({
        'ranking_diff':          ranking_diff,
        'goals_scored_avg_diff': goals_scored_diff,
        'goals_conceded_avg_diff': goals_conceded_diff,
        'win_rate_diff':         win_rate_diff,
        'head_to_head_wins':     h2h_wins,
        'tournament_stage':      stage,
        'confederation_advantage': conf_adv,
        'avg_player_age_diff':   age_diff,
        'outcome':               outcomes
    })

df_wc = generate_world_cup_data(n_matches)

outcome_names = {0: 'Team 1 Win', 1: 'Draw', 2: 'Team 2 Win'}
print("World Cup Dataset created!")
print(f"Shape: {df_wc.shape}  ({n_matches} matches, {df_wc.shape[1]-1} features + 1 label)")
print()
print("First 5 rows:")
display(df_wc.head())
print()
print("Outcome distribution:")
for k, count in df_wc['outcome'].value_counts().sort_index().items():
    print(f"  {outcome_names[k]:>12}: {count:>4} ({count/n_matches:.1%})")

In [ ]:
# ── EXPLORE THE DATA (Exploratory Data Analysis / EDA) ────────────────────────
# Before training, always understand your data visually.

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

features = [
    'ranking_diff', 'goals_scored_avg_diff', 'goals_conceded_avg_diff',
    'win_rate_diff', 'head_to_head_wins', 'tournament_stage',
    'confederation_advantage', 'avg_player_age_diff'
]

outcome_colors = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
outcome_labels = {0: 'Team 1 Win', 1: 'Draw', 2: 'Team 2 Win'}

for idx, feat in enumerate(features):
    ax = axes[idx]
    for outcome in [0, 1, 2]:
        subset = df_wc[df_wc['outcome'] == outcome][feat]
        ax.hist(subset, bins=25, alpha=0.55,
                color=outcome_colors[outcome],
                label=outcome_labels[outcome], density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    if idx == 0:
        ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Match Outcome', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Observation: features like 'ranking_diff' and 'win_rate_diff' show clear")
print("separation between outcomes — these will be powerful predictors.")

In [ ]:
# ── CORRELATION HEATMAP ────────────────────────────────────────────────────────
# See how features relate to each other and to the outcome.
# Values close to +1 or -1 mean strong relationship; close to 0 means little relationship.

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_wc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))   # Show only lower triangle

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

# Show which features correlate most with outcome
print("Correlation with outcome (absolute value):")
outcome_corr = corr['outcome'].drop('outcome').abs().sort_values(ascending=False)
for feat, val in outcome_corr.items():
    bar = '█' * int(val * 40)
    print(f"  {feat:<30} {val:.3f}  {bar}")

In [ ]:
# ── PREPARE DATA FOR TRAINING ──────────────────────────────────────────────────

feature_cols = [
    'ranking_diff', 'goals_scored_avg_diff', 'goals_conceded_avg_diff',
    'win_rate_diff', 'head_to_head_wins', 'tournament_stage',
    'confederation_advantage', 'avg_player_age_diff'
]

# X = all features (8 columns), y = outcome labels
X_wc = df_wc[feature_cols].values
y_wc = df_wc['outcome'].values

# Split: 80% train, 20% test
X_wc_train, X_wc_test, y_wc_train, y_wc_test = train_test_split(
    X_wc, y_wc, test_size=0.2, random_state=42, stratify=y_wc
)   # stratify=y_wc ensures equal proportion of outcomes in each split

# Normalise features (critical for neural networks!)
wc_scaler = StandardScaler()
X_wc_train_s = wc_scaler.fit_transform(X_wc_train)
X_wc_test_s  = wc_scaler.transform(X_wc_test)

print("Data ready for training!")
print(f"  Training samples  : {X_wc_train.shape[0]}")
print(f"  Test samples      : {X_wc_test.shape[0]}")
print(f"  Number of features: {X_wc_train.shape[1]}")
print(f"  Classes to predict: {np.unique(y_wc)} (0=Win, 1=Draw, 2=Loss)")
print()
print("Feature means after scaling (should all be near 0):")
print(X_wc_train_s.mean(axis=0).round(3))

---
# STEP 6: Designing the Neural Network Architecture

The architecture is the blueprint of the network: how many layers, how many neurons per layer.

## Our Architecture

```
Input Layer      Hidden Layer 1   Hidden Layer 2   Hidden Layer 3   Output Layer
(8 neurons)      (64 neurons)     (32 neurons)     (16 neurons)     (3 neurons)
    │                │                │                │                │
ranking_diff ──►                                                  ► Team1 Win prob
goals_scored ──►   [64 ReLU    ──► [32 ReLU     ──► [16 ReLU    ► Draw prob
    ...        ──►  neurons]        neurons]          neurons]   ► Team2 Win prob
age_diff     ──►
```

## Why these sizes?
- **More neurons = more capacity** to learn complex patterns
- **Gradually shrinking** sizes (64→32→16) is a common pattern — it compresses the information
- **3 output neurons** because we have 3 possible outcomes (Win/Draw/Loss)
- The network outputs **3 probabilities** that sum to 1.0

## Hyperparameters (settings you choose)

| Hyperparameter | Value | Meaning |
|---|---|---|
| `hidden_layer_sizes` | (64, 32, 16) | 3 hidden layers with 64, 32, 16 neurons |
| `activation` | 'relu' | Activation function |
| `learning_rate_init` | 0.001 | Step size for gradient descent |
| `max_iter` | 1000 | Maximum training epochs |
| `alpha` | 0.001 | L2 regularisation (prevents overfitting) |

In [ ]:
# ── TRAIN THE WORLD CUP NEURAL NETWORK ────────────────────────────────────────

wc_model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),   # Three hidden layers
    activation='relu',                 # ReLU activation
    solver='adam',                     # Adam optimiser (smarter than basic gradient descent)
    alpha=0.001,                       # L2 regularisation strength
    learning_rate='adaptive',          # Reduce learning rate when progress stalls
    learning_rate_init=0.001,          # Starting learning rate
    max_iter=1000,                     # Up to 1000 epochs
    early_stopping=True,               # Stop if validation score stops improving
    validation_fraction=0.15,          # Use 15% of training data for validation
    n_iter_no_change=30,               # Stop after 30 epochs with no improvement
    random_state=42,
    verbose=False
)

print("Training the World Cup Neural Network...")
wc_model.fit(X_wc_train_s, y_wc_train)

print(f"Training stopped after {wc_model.n_iter_} epochs (early stopping kicked in)")
print(f"Final training loss: {wc_model.loss_:.4f}")

In [ ]:
# ── PLOT THE ACTUAL TRAINING LOSS CURVE ───────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(wc_model.loss_curve_, color='steelblue', linewidth=2, label='Training loss')

if hasattr(wc_model, 'validation_scores_') and wc_model.validation_scores_ is not None:
    # Plot validation score (inverted so it looks like a loss)
    val_loss = [1 - s for s in wc_model.validation_scores_]
    ax.plot(val_loss, color='orange', linewidth=2, linestyle='--', label='Validation loss')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('World Cup Model: Training Loss Curve', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("If validation loss starts rising while training loss keeps falling,")
print("that's overfitting — the model is memorising instead of learning.")
print("Early stopping prevents this automatically.")

In [ ]:
# ── EVALUATE ON THE TEST SET ───────────────────────────────────────────────────

y_wc_pred = wc_model.predict(X_wc_test_s)
y_wc_proba = wc_model.predict_proba(X_wc_test_s)   # Probability of each outcome

test_acc = accuracy_score(y_wc_test, y_wc_pred)

# Cross-validation: train/test on 5 different splits and average
cv_scores = cross_val_score(wc_model, wc_scaler.transform(X_wc), y_wc, cv=5, scoring='accuracy')

print("=" * 50)
print("   WORLD CUP MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"Test Set Accuracy   : {test_acc:.1%}")
print(f"Cross-Val Accuracy  : {cv_scores.mean():.1%} ± {cv_scores.std():.1%}")
print()
print("Breakdown by outcome:")
print(classification_report(
    y_wc_test, y_wc_pred,
    target_names=['Team 1 Win', 'Draw', 'Team 2 Win']
))

# ── Baseline comparison ───────────────────────────────────────────────────────
# A 'dumb' model that always predicts the most common outcome
most_common = np.bincount(y_wc_test).argmax()
baseline_acc = (y_wc_test == most_common).mean()
print(f"Baseline (always predict most common): {baseline_acc:.1%}")
print(f"Our model improvement over baseline  : +{(test_acc - baseline_acc):.1%}")

In [ ]:
# ── CONFUSION MATRIX FOR WORLD CUP MODEL ──────────────────────────────────────

cm_wc = confusion_matrix(y_wc_test, y_wc_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_wc, annot=True, fmt='d', cmap='YlOrRd',
    xticklabels=['Team 1 Win', 'Draw', 'Team 2 Win'],
    yticklabels=['Team 1 Win', 'Draw', 'Team 2 Win'],
    ax=ax, linewidths=0.5
)
ax.set_xlabel('Predicted Outcome', fontsize=12)
ax.set_ylabel('Actual Outcome',    fontsize=12)
ax.set_title('World Cup Model — Confusion Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

print("The diagonal (top-left to bottom-right) shows CORRECT predictions.")
print("Off-diagonal cells show mistakes. Notice 'Draw' is hardest to predict —")
print("this is true in real football prediction too!")

---
# STEP 7: Making Actual Predictions — The Fun Part!

Now let's use our trained model to predict specific World Cup match-ups.

We'll create a `predict_match` function that:
1. Takes readable team stats as input
2. Runs them through the neural network
3. Returns human-friendly probabilities

In [ ]:
# ── TEAM DATABASE ─────────────────────────────────────────────────────────────
# Real-world inspired stats (based on historical averages)
# Format: name, fifa_ranking, avg_goals_scored, avg_goals_conceded, win_rate

teams = {
    # Name                   rank  gs_avg  gc_avg  win_rate
    'Brazil':               (  1,   2.10,   0.80,   0.72),
    'Argentina':            (  2,   2.05,   0.85,   0.70),
    'France':               (  3,   1.95,   0.90,   0.68),
    'England':              (  4,   1.85,   0.95,   0.65),
    'Spain':                (  5,   1.90,   0.88,   0.67),
    'Germany':              (  6,   1.88,   0.92,   0.66),
    'Netherlands':          (  7,   1.80,   0.95,   0.63),
    'Portugal':             (  8,   1.85,   0.98,   0.64),
    'Belgium':              (  9,   1.78,   0.98,   0.63),
    'Italy':                ( 10,   1.72,   0.88,   0.62),
    'Croatia':              ( 12,   1.65,   1.02,   0.58),
    'Morocco':              ( 13,   1.55,   1.05,   0.57),
    'USA':                  ( 14,   1.52,   1.10,   0.55),
    'Mexico':               ( 15,   1.48,   1.12,   0.54),
    'Japan':                ( 20,   1.42,   1.15,   0.50),
    'Senegal':              ( 22,   1.38,   1.18,   0.48),
    'Australia':            ( 27,   1.22,   1.35,   0.42),
    'South Korea':          ( 28,   1.20,   1.38,   0.41),
    'Ghana':                ( 55,   1.10,   1.45,   0.38),
    'Saudi Arabia':         ( 58,   1.05,   1.55,   0.35),
    'Cameroon':             ( 42,   1.15,   1.42,   0.40),
    'Ecuador':              ( 44,   1.12,   1.48,   0.38),
    'Wales':                ( 18,   1.45,   1.20,   0.49),
    'Switzerland':          ( 16,   1.50,   1.08,   0.52),
}

print(f"Team database loaded: {len(teams)} teams")
print()
print(f"{'Team':<20} {'FIFA Rank':>10} {'Goals/G':>9} {'Conceded/G':>12} {'Win Rate':>10}")
print("-" * 65)
for team, (rank, gs, gc, wr) in sorted(teams.items(), key=lambda x: x[1][0])[:10]:
    print(f"{team:<20} {rank:>10} {gs:>9.2f} {gc:>12.2f} {wr:>9.1%}")
print("... (showing top 10 by ranking)")

In [ ]:
# ── PREDICTION ENGINE ─────────────────────────────────────────────────────────

def predict_match(team1_name, team2_name, stage=0, confederation_advantage=0,
                  h2h_wins_team1=2, age_diff=0.0, verbose=True):
    """
    Predict the outcome of a World Cup match.

    Parameters:
    -----------
    team1_name            : str  — Name of team 1 (must be in teams database)
    team2_name            : str  — Name of team 2
    stage                 : int  — 0=Group, 1=R16, 2=QF, 3=SF, 4=Final
    confederation_advantage: int — +1 if team 1 on home continent, -1 if team 2, 0 if neither
    h2h_wins_team1        : int  — Team 1's head-to-head wins out of last 5 (0–5)
    age_diff              : float— Average age difference (team1 age - team2 age)
    verbose               : bool — Print formatted output

    Returns: dict with probabilities
    """
    if team1_name not in teams:
        raise ValueError(f"Team '{team1_name}' not in database. Options: {list(teams.keys())}")
    if team2_name not in teams:
        raise ValueError(f"Team '{team2_name}' not in database. Options: {list(teams.keys())}")

    r1, gs1, gc1, wr1 = teams[team1_name]
    r2, gs2, gc2, wr2 = teams[team2_name]

    # Compute feature vector (same order as training!)
    features = np.array([[
        r2 - r1,                  # ranking_diff: positive = team1 ranked higher
        gs1 - gs2,                # goals_scored_avg_diff
        gc2 - gc1,                # goals_conceded_avg_diff (positive = team1 better defence)
        wr1 - wr2,                # win_rate_diff
        h2h_wins_team1,           # head_to_head_wins
        stage,                    # tournament_stage
        confederation_advantage,  # confederation_advantage
        age_diff,                 # avg_player_age_diff
    ]])

    # Scale using the SAME scaler fitted on training data
    features_scaled = wc_scaler.transform(features)

    # Get probabilities for each outcome
    proba = wc_model.predict_proba(features_scaled)[0]
    prediction = wc_model.predict(features_scaled)[0]

    outcome_map = {0: f'{team1_name} Win', 1: 'Draw', 2: f'{team2_name} Win'}
    result = {
        'team1_win_prob': proba[0],
        'draw_prob':      proba[1],
        'team2_win_prob': proba[2],
        'predicted_outcome': outcome_map[prediction]
    }

    if verbose:
        stage_names = ['Group Stage', 'Round of 16', 'Quarter-Final', 'Semi-Final', 'Final']
        print(f"{'=' * 52}")
        print(f"  {team1_name:>20}  vs  {team2_name:<20}")
        print(f"  Stage: {stage_names[stage]}")
        print(f"{'=' * 52}")
        print(f"  {team1_name} Win  : {proba[0]:>6.1%}  {'█' * int(proba[0]*30)}")
        print(f"  Draw       : {proba[1]:>6.1%}  {'█' * int(proba[1]*30)}")
        print(f"  {team2_name} Win  : {proba[2]:>6.1%}  {'█' * int(proba[2]*30)}")
        print(f"{'─' * 52}")
        print(f"  PREDICTED: {result['predicted_outcome'].upper()}")
        print(f"{'=' * 52}")

    return result

print("Prediction engine ready!")

In [ ]:
# ── RUN PREDICTIONS FOR FAMOUS MATCH-UPS ──────────────────────────────────────

print("WORLD CUP MATCH PREDICTIONS")
print()

# Classic rivalry: Brazil vs Argentina (Semi-Final)
r1 = predict_match('Brazil', 'Argentina', stage=3)
print()

# Dark horse: Morocco vs Spain (Quarter-Final)
r2 = predict_match('Morocco', 'Spain', stage=2, confederation_advantage=-1)  # Spain has European advantage
print()

# Heavyweight clash: France vs England (Final)
r3 = predict_match('France', 'England', stage=4)
print()

# Giant killing: Japan vs Germany (Group Stage)
r4 = predict_match('Japan', 'Germany', stage=0, h2h_wins_team1=1)

In [ ]:
# ── SIMULATE A FULL KNOCKOUT BRACKET ──────────────────────────────────────────

def simulate_bracket(bracket):
    """
    Simulate a knockout tournament bracket.

    bracket: list of (team1, team2) tuples for each round.
    Returns the winner.
    """
    rounds = ['Round of 16', 'Quarter-Finals', 'Semi-Finals', 'Final']
    stage_map = {'Round of 16': 1, 'Quarter-Finals': 2, 'Semi-Finals': 3, 'Final': 4}

    current_teams = bracket[:]
    round_idx = 0

    while len(current_teams) > 1:
        round_name = rounds[round_idx] if round_idx < len(rounds) else 'Extra Round'
        stage = stage_map.get(round_name, 1)

        print(f"\n{'─'*50}")
        print(f"  {round_name}")
        print(f"{'─'*50}")

        winners = []
        pairs = [(current_teams[i], current_teams[i+1])
                 for i in range(0, len(current_teams), 2)]

        for t1, t2 in pairs:
            res = predict_match(t1, t2, stage=stage, verbose=False)
            probs = [
                (t1, res['team1_win_prob'] + res['draw_prob'] * 0.5),
                (t2, res['team2_win_prob'] + res['draw_prob'] * 0.5)
            ]
            # Use probabilities to determine winner (deterministic: pick highest prob)
            winner = max(probs, key=lambda x: x[1])[0]
            t1_win_p = res['team1_win_prob']
            t2_win_p = res['team2_win_prob']
            draw_p   = res['draw_prob']
            print(f"  {t1:<15} {t1_win_p:.0%} | Draw {draw_p:.0%} | {t2_win_p:.0%} {t2:<15}  → {winner}")
            winners.append(winner)

        current_teams = winners
        round_idx += 1

    return current_teams[0]

# Our simulated bracket (top 8 teams)
bracket_r16 = [
    'Brazil', 'South Korea',
    'France', 'Poland',
    'Argentina', 'Australia',
    'England', 'Senegal',
]  # Must have 2^n teams

# Quick-add Poland and Poland since not in DB
teams['Poland']   = (26, 1.38, 1.20, 0.48)

print("KNOCKOUT BRACKET SIMULATION")
champion = simulate_bracket(bracket_r16)
print(f"\n{'='*50}")
print(f"  PREDICTED WORLD CUP CHAMPION: {champion.upper()}")
print(f"{'='*50}")

---
# STEP 8: How to Improve the Model Over Time

Neural networks rarely work perfectly on the first try. This section covers systematic techniques to boost performance.

## The Improvement Toolkit

| Technique | What it does | When to use |
|-----------|-------------|-------------|
| **More data** | More training examples | Almost always helps |
| **Better features** | Add domain knowledge | When accuracy is stuck |
| **Larger network** | More neurons/layers | When model underfits |
| **Regularisation** | Penalise complexity | When model overfits |
| **Dropout** | Randomly disable neurons | Prevents overfitting |
| **Learning rate tuning** | Smaller/larger steps | When loss isn't decreasing |
| **Hyperparameter search** | Try many settings | Systematic optimisation |

In [ ]:
# ── DEMO: OVERFITTING vs UNDERFITTING vs JUST RIGHT ───────────────────────────
# This is one of the most important concepts in machine learning.

architectures = {
    'Underfitting (too small)':   (4,),          # Only 4 neurons — can't learn patterns
    'Good fit':                   (64, 32, 16),   # Our main model
    'Overfitting (too large)':    (512, 256, 128, 64),  # Massive — memorises instead of learning
}

results = {}

for name, arch in architectures.items():
    model = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=500,
        random_state=42,
        alpha=0.0001,    # Very little regularisation for the demo
        verbose=False
    )
    model.fit(X_wc_train_s, y_wc_train)

    train_acc = model.score(X_wc_train_s, y_wc_train)
    test_acc  = model.score(X_wc_test_s,  y_wc_test)
    gap = train_acc - test_acc

    results[name] = {'train': train_acc, 'test': test_acc, 'gap': gap}
    print(f"{name}")
    print(f"  Architecture: {arch}")
    print(f"  Train accuracy: {train_acc:.1%}")
    print(f"  Test  accuracy: {test_acc:.1%}")
    print(f"  Gap (overfit indicator): {gap:.1%}")
    if gap > 0.05:
        print(f"  ⚠ Large gap = overfitting!")
    elif train_acc < 0.5:
        print(f"  ⚠ Low train accuracy = underfitting!")
    else:
        print(f"  ✓ Balanced — good generalisation")
    print()

In [ ]:
# ── VISUALISE OVERFIT / UNDERFIT / GOOD FIT ───────────────────────────────────

names = list(results.keys())
train_accs = [results[n]['train'] for n in names]
test_accs  = [results[n]['test']  for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, test_accs,  width, label='Test Accuracy',  color='coral',     alpha=0.85)

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Underfitting vs Good Fit vs Overfitting', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([n.split('(')[0].strip() for n in names], fontsize=11)
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=11)
ax.axhline(0.5, color='gray', linewidth=1, linestyle='--', alpha=0.5)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("Key insight:")
print("- Underfitting: both train AND test accuracy are low")
print("- Overfitting:  train accuracy is high but test accuracy drops")
print("- Good fit:     both are similar and reasonably high")

In [ ]:
# ── HYPERPARAMETER SEARCH ─────────────────────────────────────────────────────
# Try different settings and see which works best.
# This is called 'hyperparameter tuning'.

from sklearn.model_selection import GridSearchCV

# Define the settings we want to try
param_grid = {
    'hidden_layer_sizes': [(32, 16), (64, 32), (64, 32, 16)],
    'alpha':              [0.0001, 0.001, 0.01],   # Regularisation strength
    'learning_rate_init': [0.001, 0.005],
}

# GridSearchCV tries every combination using cross-validation
base_model = MLPClassifier(max_iter=300, activation='relu',
                           solver='adam', random_state=42)

print("Running hyperparameter search (this may take ~30 seconds)...")
grid_search = GridSearchCV(
    base_model, param_grid,
    cv=3,                # 3-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,           # Use all CPU cores
    verbose=0
)

grid_search.fit(X_wc_train_s, y_wc_train)

print(f"\nBest parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest cross-val accuracy: {grid_search.best_score_:.1%}")

best_model = grid_search.best_estimator_
best_test_acc = best_model.score(X_wc_test_s, y_wc_test)
print(f"Best model test accuracy: {best_test_acc:.1%}")
print(f"Improvement over original: +{(best_test_acc - test_acc):.1%}")

In [ ]:
# ── FEATURE IMPORTANCE (via Permutation) ──────────────────────────────────────
# Which features matter most? We shuffle each feature and see how much accuracy drops.
# Big drop = feature was important.

from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(
    wc_model, X_wc_test_s, y_wc_test,
    n_repeats=20, random_state=42, n_jobs=-1
)

sorted_idx = perm_imp.importances_mean.argsort()[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
colors_feat = sns.color_palette('Set2', len(feature_cols))
bars = ax.barh(
    [feature_cols[i].replace('_', ' ').title() for i in sorted_idx],
    perm_imp.importances_mean[sorted_idx],
    xerr=perm_imp.importances_std[sorted_idx],
    color=[colors_feat[i] for i in sorted_idx],
    alpha=0.85
)
ax.set_xlabel('Mean Accuracy Decrease When Feature is Shuffled', fontsize=11)
ax.set_title('Feature Importance (Permutation Method)', fontsize=13)
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("Most important features (largest accuracy drop when removed):")
for i in sorted_idx[:4]:
    print(f"  {feature_cols[i]:<35} importance: {perm_imp.importances_mean[i]:.4f}")

In [ ]:
# ── PROBABILITY CALIBRATION CHECK ─────────────────────────────────────────────
# A well-calibrated model: when it says 70% chance of winning, teams win ~70% of the time.
# This matters more than raw accuracy for sports betting or decision making.

from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
class_names = ['Team 1 Win', 'Draw', 'Team 2 Win']

for class_idx in range(3):
    # Binarise: is the outcome this class or not?
    y_binary = (y_wc_test == class_idx).astype(int)
    prob_pos  = y_wc_proba[:, class_idx]

    if y_binary.sum() > 10:   # Need enough positive examples
        fraction_pos, mean_pred = calibration_curve(
            y_binary, prob_pos, n_bins=8, strategy='quantile'
        )
        ax = axes[class_idx]
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
        ax.plot(mean_pred, fraction_pos, 's-', color='steelblue',
                linewidth=2, markersize=6, label='Our model')
        ax.set_xlabel('Predicted Probability', fontsize=10)
        ax.set_ylabel('Actual Fraction',        fontsize=10)
        ax.set_title(f'Calibration: {class_names[class_idx]}', fontsize=11)
        ax.legend(fontsize=9)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('Calibration Curves — How Reliable Are the Probabilities?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Perfect calibration = following the diagonal dashed line.")
print("Our model is reasonably calibrated — the probabilities are meaningful.")

---
# STEP 9: Where to Go from Here

You've built a working neural network! Here are pathways to take it further.

## Immediate Improvements You Can Make

1. **Add real data** — Download actual World Cup results from [Kaggle](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)
2. **Add more features** — Market value of squad (Transfermarkt), weather, referee, injury news
3. **Use time series** — Weight recent matches more than old ones using exponential decay
4. **Add Elo ratings** — A more sophisticated ranking system used in chess and football

## Next Machine Learning Topics

| Topic | What it adds |
|-------|-------------|
| **Deep Learning (PyTorch/TensorFlow)** | Much larger networks, GPU training |
| **Ensemble methods (XGBoost, Random Forest)** | Often beats neural nets on tabular data |
| **LSTM / Transformer** | Models sequential match history |
| **Poisson regression** | Specifically models score distributions |
| **Bayesian models** | Outputs uncertainty ranges, not just point predictions |

## Free Resources
- [fast.ai](https://www.fast.ai) — Free practical deep learning course
- [Kaggle Learn](https://www.kaggle.com/learn) — Hands-on ML micro-courses
- [3Blue1Brown Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi) — Beautiful visual explanations
- [StatsBomb Open Data](https://github.com/statsbomb/open-data) — Real football event data

In [ ]:
# ── INTERACTIVE MATCH PREDICTOR ───────────────────────────────────────────────
# Try any match-up from the team database!

print("AVAILABLE TEAMS:")
team_list = sorted(teams.keys())
for i, t in enumerate(team_list):
    print(f"  {t}", end="\n" if (i+1) % 3 == 0 else "\t\t")

print("\n\n" + "=" * 55)
print("EXAMPLE PREDICTIONS — CHANGE THESE TO TRY YOUR OWN:")
print("=" * 55)

# Change team names here to predict any match!
TEAM_1 = 'Germany'
TEAM_2 = 'Italy'
STAGE  = 2   # 0=Group, 1=R16, 2=QF, 3=SF, 4=Final

predict_match(TEAM_1, TEAM_2, stage=STAGE)

In [ ]:
# ── SIMULATE A FULL GROUP STAGE ────────────────────────────────────────────────

def simulate_group(group_name, group_teams):
    """Simulate a World Cup group stage and print the standings table."""
    print(f"\n{'═'*60}")
    print(f"  GROUP {group_name} — All matches")
    print(f"{'═'*60}")

    standings = {t: {'pts': 0, 'w': 0, 'd': 0, 'l': 0, 'gf': 0, 'ga': 0}
                 for t in group_teams}

    # Every team plays every other team once (round-robin)
    for i, t1 in enumerate(group_teams):
        for t2 in group_teams[i+1:]:
            res = predict_match(t1, t2, stage=0, verbose=False)
            p1 = res['team1_win_prob']
            pd_val = res['draw_prob']
            p2 = res['team2_win_prob']

            # Award points based on predicted probabilities (expected points)
            pts1 = 3 * p1 + 1 * pd_val
            pts2 = 3 * p2 + 1 * pd_val

            print(f"  {t1:<15} {p1:.0%} | Draw {pd_val:.0%} | {p2:.0%} {t2:<15}")

            standings[t1]['pts'] += pts1
            standings[t2]['pts'] += pts2

            if p1 > p2: standings[t1]['w'] += 1; standings[t2]['l'] += 1
            elif p2 > p1: standings[t2]['w'] += 1; standings[t1]['l'] += 1
            else: standings[t1]['d'] += 1; standings[t2]['d'] += 1

    # Sort by expected points
    sorted_standings = sorted(standings.items(), key=lambda x: x[1]['pts'], reverse=True)

    print(f"\n  FINAL STANDINGS:")
    print(f"  {'Pos':<5}{'Team':<18}{'Exp Pts':>8}{'W':>4}{'D':>4}{'L':>4}")
    print(f"  {'─'*43}")
    for pos, (team, stats) in enumerate(sorted_standings, 1):
        qualifier = "← QUALIFIES" if pos <= 2 else ""
        print(f"  {pos:<5}{team:<18}{stats['pts']:>7.1f}{stats['w']:>4}{stats['d']:>4}{stats['l']:>4}  {qualifier}")

    return [s[0] for s in sorted_standings[:2]]

# Simulate Group of Death!
group_qualifiers = simulate_group('E (Group of Death)', ['Brazil', 'France', 'Spain', 'Japan'])

---
# Congratulations — You've Completed the Tutorial!

## What You Built

| Component | Description |
|-----------|-------------|
| Single neuron | Hand-coded from scratch with sigmoid activation |
| Warm-up classifier | Pass/fail predictor with decision boundary visualisation |
| World Cup model | 8-feature, 3-class neural network with 1500 training samples |
| Prediction engine | Function that gives probabilities for any match-up |
| Bracket simulator | Simulates full knockout tournaments |
| Group stage simulator | Simulates round-robin groups with standings |

## Key Concepts You Learned

| Concept | One-Line Summary |
|---------|------------------|
| Neurons & weights | Building blocks; weights determine feature importance |
| Activation functions | Non-linearity that lets networks learn complex patterns |
| Forward pass | Data flows input → hidden → output to produce a prediction |
| Loss function | Measures how wrong the prediction is |
| Backpropagation | Calculates how to adjust each weight to reduce loss |
| Gradient descent | Strategy of always moving weights toward lower loss |
| Training vs test split | Train on one set, evaluate on a completely unseen set |
| Normalisation | Scale inputs so the network learns faster and more reliably |
| Overfitting | Memorising training data instead of generalising |
| Hyperparameter tuning | Systematically searching for the best model settings |
| Feature importance | Finding which inputs the model relies on most |

## The Full Neural Network Pipeline

```
Raw Data → Feature Engineering → Normalise → Train/Test Split
                                                      │
                                              Train Neural Network
                                                      │
                                          Evaluate on Test Set
                                                      │
                                    Is accuracy good enough?
                                      /                \
                                    YES                 NO
                                     │                   │
                             Deploy & predict      Tune hyperparameters
                                                   Add more data/features
                                                   Try different architecture
```

---

*Keep experimenting, keep learning, and may your predictions be correct!*